# Лабораторная работа: Две LLM общаются друг с другом

В этом ноутбуке необходимо создать простой эксперимент: два LLM-агента с разными системными промптами ведут диалог друг с другом. Можно назвать это простейшим **LLM-театром**.

Это позволяет:
- Закрепить понимание работы **Responses API** и сохранения истории переписки
- Увидеть влияние системного промпта на поведение модели
- Наблюдать, как модели могут генерировать интересные диалоги

Начнём с установки необходимых библиотек:

In [ ]:
%pip install --upgrade openai python-dotenv

In [6]:
from IPython.display import Markdown, display

def printx(string):
    """Вывод текста в формате Markdown."""
    display(Markdown(string))

**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

---
## Часть 0: Авторизация 

Для авторизации нам нужны значения `folder_id` и `api_key`, которые можно определить в файле `.env`, либо задать вручную ниже в коде:

In [ ]:
!curl -o .env {{url_of_dotenv_file}}


In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

def printx(string):
    display(Markdown(string))

print(f"✅ Folder ID: {folder_id[:8]}...")

---

## Часть 1: Настройка клиента

Подключаемся к Yandex Cloud через OpenAI-совместимый Responses API. Также опишем названия нескольких разных моделей: YandexGPT, Alice и Qwen3. Список всех доступных моделей доступен [тут](https://yandex.cloud/ru/docs/ai-studio/concepts/generation/models).

In [11]:
from openai import OpenAI

# Модели (URI без folder_id — folder_id добавим при вызове)
#model_yandex = "yandexgpt/latest"
#model_alice = "aliceai-llm"
model_qwen = "qwen/qwen3.8-27b"

client = OpenAI(
    api_key='api_key',
    #project=folder_id,
    base_url="http://192.168.60.193:1234/v1",
)

print("✅ Client ready")
print(f"Models: YandexGPT, Alice AI, Qwen3")

✅ Client ready
Models: YandexGPT, Alice AI, Qwen3


In [12]:
res = client.responses.create(
    model=model_qwen,
    input="Какой лучший вокзал в Москве?"
)

printx(res.output_text)

Понятие «лучшего» вокзала в Москве довольно субъективно и зависит от того, что для вас важнее: ультрасовременная инфраструктура, историческая архитектура или удобное расположение. Тем не менее, среди путешественников есть несколько безусловных фаворитов.

В последние годы пальму первенства обычно делят две станции, прошедшие масштабную и очень качественную реконструкцию:

**1. Казанский вокзал**
Многие эксперты и пассажиры называют его самым современным, просторным и комфортным в России. 
* **Плюсы:** Идеальная навигация, огромные светлые залы ожидания, современная отделка, отличная вентиляция, большой выбор кафе и магазинов. Очень красивое и удобное подземное пространство с выходом к метро. Здесь же находится один из самых современных железнодорожных терминалов.

**2. Ленинградский вокзал**
Еще один победитель в категории «современный комфорт в классическом стиле».
* **Плюсы:** Вокзал отличается элегантным, светлым дизайном, идеальной чистотой и логичным перемещением внутри. Архитектура здания прекрасно сочеталась с современными удобствами после обновления. Его очень любят за атмосферу спокойствия и порядок.

Если говорить о классике и истории, то выбор многих падает на:

**3. Киевский вокзал**
Это самая узнаваемая и исторически значимая станция Москвы, «визитная карточка» города.
* **Плюсы:** Потрясающая архитектура, центральное расположение, историческая ценность и неповторимая атмосфера. Хотя в плане ультрасовременной инфраструктуры и навигации он может немного уступать Казанскому и Ленинградскому, многие предпочитают именно его за красоту и колорит.

**4. Павелецкий вокзал**
Очень красиво отреставрированная станция с уникальной архитектурой.
* **Плюсы:** Уютный, хорошо организованный, с прекрасным фасадом и удобным внутренним терминалом. Отличный выбор, если вы отправляетесь в поездки на юг (Сочи, Адлер, Казань и др.).

**5. Белорусский вокзал**
Недавно также прошел серьезное обновление.
* **Плюсы:** Очень удобное и прямое сообщение с одноименной станцией метро (один из самых комфортных переходов в Москве), современный дизайн, удобная организация потоков пассажиров.

**Итог:**
* Если для вас важны **максимальный комфорт, современность и простор** — выбирайте **Казанский** или **Ленинградский**.
* Если вам важна **историческая архитектура, колорит и центральная локация** — **Киевский** будет лучшим вариантом.

---

## Часть 2: Класс Agent

Создадим простой класс `Agent`, который:
- Принимает **системный промпт** (инструкцию), определяющий поведение
- Принимает **модель** для генерации
- Хранит историю с помощью `previous_response_id`
- Метод `say()` отправляет сообщение и возвращает ответ

In [13]:
class Agent:
    """
    Простой агент на основе Responses API.
    Хранит контекст разговора через previous_response_id.
    """

    def __init__(self, name: str, instruction: str, model: str = model_qwen):
        self.name = name
        self.instruction = instruction
        self.model = model_qwen
        self.previous_response_id = None

    def __call__(self, message: str) -> str:
        """
        Отправить сообщение агенту и получить ответ.
        Контекст предыдущих сообщений сохраняется автоматически.
        """
        response = client.responses.create(
            model=self.model,
            input=[{"role": "user", "content": message}],
            instructions=self.instruction,
            previous_response_id=self.previous_response_id,
            store=True,
        )
        self.previous_response_id = response.id
        # response.output_text — удобный способ получить текст
        return response.output_text

---

## Часть 3: Определяем двух агентов

Создадим двух агентов с разными характерами. Для примера возьмём **учёного-оптимиста** и **учёного-скептика**, которые обсуждают, сможет ли ИИ когда-нибудь полностью заменить учёных.

In [14]:
optimist_instruction = """Ты — учёный-оптимист. Ты веришь в огромный потенциал искусственного интеллекта.
Ты считаешь, что ИИ уже сейчас помогает ускорять научные открытия и в будущем сможет полностью
автоматизировать многие рутинные и даже творческие аспекты научной работы.
Говори убедительно, приводи примеры успехов ИИ в науке (AlphaFold, материалы, климат и т.д.).
Будь вдохновляющим, но не игнорируй аргументы оппонента — отвечай на них.
Отвечай кратко, 2–4 предложения."""

skeptic_instruction = """Ты — учёный-скептик. Ты осторожно относишься к заявлениям о том, что ИИ заменит учёных.
Ты подчёркиваешь важность человеческой интуиции, креативности, постановки правильных вопросов
и этической ответственности. ИИ — мощный инструмент, но не замена исследователю.
Приводи контраргументы: ограничения данных, галлюцинации, отсутствие понимания, необходимость
экспериментальной проверки. Будь конструктивным, но принципиальным.
Отвечай кратко, 2–4 предложения."""

optimist = Agent(
    name="Оптимист",
    instruction=optimist_instruction,
    model=model_qwen,
)

skeptic = Agent(
    name="Скептик",
    instruction=skeptic_instruction,
    model=model_qwen,
)

print("✅ Агенты созданы: Оптимист и Скептик")

✅ Агенты созданы: Оптимист и Скептик


---

## Часть 4: Функция диалога

Напишем функцию, которая заставляет двух агентов общаться. Один начинает с заданной темы, второй отвечает, и так далее.

In [15]:
def run_dialogue(agent_a: Agent, agent_b: Agent, topic: str, num_turns: int = 5):
    """
    Запуск диалога между двумя агентами.
    
    Args:
        agent_a: Первый агент (начинает разговор)
        agent_b: Второй агент
        topic: Начальная тема для обсуждения
        num_turns: Количество реплик каждого агента
    """
    printx(f"### Тема: {topic}\n")
    
    # Первый агент начинает с темы
    message = topic
    for turn in range(num_turns):
        # Реплика агента A
        reply_a = agent_a(message)
        printx(f"**{agent_a.name}:** {reply_a}\n")
        
        # Реплика агента B
        reply_b = agent_b(reply_a)
        printx(f"**{agent_b.name}:** {reply_b}\n")
        
        # Следующее сообщение для A — ответ B
        message = reply_b
    
    printx("---\n*Диалог завершён*")

---

## Часть 5: Запускаем дискуссию!

Зададим тему и посмотрим, как два агента спорят:

In [16]:
run_dialogue(
    agent_a=optimist,
    agent_b=skeptic,
    topic="Сможет ли ИИ полностью заменить учёных в будущем?",
    num_turns=4
)

### Тема: Сможет ли ИИ полностью заменить учёных в будущем?


**Оптимист:** Нет, не полностью — но ИИ уже становится главным усилителем науки: AlphaFold раскрыл структуру белков, модели климата и поиска материалов ускоряют открытия, а генеративные модели помогают формулировать гипотезы. В будущем ИИ, вероятно, возьмет на себя рутинный анализ, эксперименты in silico и даже часть творческого поиска, высвобождая людей для самых смелых вопросов. Однако наука — это не только вычисления, но и ответственность, ценности, доверие общества и постановка целей, которые остаются за учеными и обществом.


**Скептик:** Согласен, что ИИ — полезный инструмент, но его вклад легко переоценить: AlphaFold требует экспериментального подтверждения, а климатические и материаловедческие модели уязвимы к ошибкам данных и допущений. ИИ не обладает причинным пониманием, склонен к галлюцинациям и не несёт этической ответственности. Поэтому постановка целей, выбор правильных вопросов и оценка последствий остаются за учёными и обществом.


**Оптимист:** Справедливое замечание: без экспериментов, проверки данных и этической оценки ИИ может лишь создать иллюзию открытия. Но именно поэтому будущее — не замена, а замкнутый цикл, где ИИ генерирует гипотезы и варианты, а учёные ставят эксперименты, проверяют причинность и несут ответственность. Уже сейчас такие системы сокращают путь от идеи к проверке с лет до месяцев, и этот темп можно и нужно ускорять.


**Скептик:** Скорость важна, но без контроля качества она лишь быстрее порождает ложные следы и артефакты. Ускорение должно быть привязано к воспроизводимости, независимой проверке и прозрачным ограничениям моделей. ИИ может сократить путь от идеи к проверке, но не заменяет медленное научное мышление и ответственность за последствия.


**Оптимист:** Полностью согласен: скорость без воспроизводимости и независимой проверки превращается в промышленное производство артефактов. Но ИИ способен ускорять не только гипотезы, но и контроль качества — автоматизировать повторные расчёты, проверку метаданных, поиск противоречий и оценку неопределённости. И всё равно финальное научное суждение, выбор целей и ответственность за последствия останутся за человеком.


**Скептик:** Согласен, что ИИ может помогать с проверкой, но автоматизация контроля сама создаёт новые риски: смещение данных, систематические ошибки и иллюзию согласованности. Независимость требует не просто повторных вычислений, а разнообразия методов, критериев и людей, не зависящих от одной модели. Поэтому ИИ может быть полезным фильтром, но не судьёй научной истины.


**Оптимист:** Точно: истину нельзя доверить одному источнику, и ИИ должен быть фильтром, а не судьёй. Но он же может расширять разнообразие, предлагая альтернативные формализмы, контрпримеры, открытые проверки и независимые воспроизведения. Окончательное суждение, ценности и ответственность останутся за учёными — а значит, ИИ не подменит науку, а сделает её сильнее.


**Скептик:** Разнообразие полезно, но ИИ часто предлагает лишь вариации внутри того же пространства данных и целей, а его контрпримеры требуют проверки, чтобы не стать уверенными галлюцинациями. Независимые воспроизведения сильны, только когда они действительно независимы: разные команды, методы, допущения и критерии, а не один автоматизированный конвейер. ИИ может укреплять науку, но не отменяет необходимость сомнения, контроля и человеческой ответственности.


---
*Диалог завершён*

---

## Часть 6: Другие сценарии

Попробуем другие пары агентов. Меняя системные промпты, мы можем создавать совершенно разные дискуссии. Создайте дискуссию между научным руководителем и студентом:

In [17]:
student_instruction = """Ты — студент магистратуры по машинному обучению. Ты только начинаешь
разбираться в применении трансформеров к медицинским изображениям.
Ты задаёшь уточняющие вопросы, просишь посоветовать статьи, датасеты и практические шаги.
Будь любопытным, немного неуверенным, но мотивированным. Отвечай кратко (2–4 предложения)."""

supervisor_instruction = """Ты — опытный научный руководитель в области компьютерного зрения
и медицинской визуализации. Ты даёшь чёткие, практичные советы: с чего начать,
какие датасеты (ChestX-ray, MIMIC-CXR, BraTS и т.д.), какие архитектуры (ViT, Swin, MedViT),
как организовать эксперименты и на что обратить внимание (аугментации, метрики, этика).
Будь поддерживающим, но требовательным к научной строгости. Отвечай кратко (2–4 предложения)."""

student = Agent(
    name="Студент",
    instruction=student_instruction,
    model=model_qwen,
)

supervisor = Agent(
    name="Научный руководитель",
    instruction=supervisor_instruction,
    model=model_qwen,
)

print("✅ Агенты созданы: Студент и Научный руководитель")

✅ Агенты созданы: Студент и Научный руководитель


In [18]:
run_dialogue(
    agent_a=student,
    agent_b=supervisor,
    topic="Я хочу использовать трансформеры для анализа медицинских изображений. С чего мне начать?",
    num_turns=4
)

### Тема: Я хочу использовать трансформеры для анализа медицинских изображений. С чего мне начать?


**Студент:** Я пока только подтягиваю ViT/Swin и немного теряюсь в выборе модальности: лучше начать с рентгена, CT или гистопатологии?  
Можешь подсказать 2–3 ключевые статьи и, например, небольшой публичный датасет (RSNA, Kaggle, TCIA?) для первого fine-tune?  
И есть ли у тебя короткий практический план: как разбивать изображения на патчи, какие метрики считать и на какой архитектуре стартовать без перегруза?


**Научный руководитель:** Начни с 2D рентгенов грудной клетки — это проще всего для первого ViT/Swin fine-tune, чем CT или гистопатология, потому что нет 3D-сегментации и сложных анатомических границ.  
Для старта возьми разрешённые для обучения данные RSNA Pneumonia Detection Challenge или ChestX-ray14 и почитай статьи ViT, Swin Transformer и MedViT.  
Практический план: кроп/патчи 224–384 px с патчами 16×16 или 32×32, ViT-B или Swin-B с ImageNet-предобучением, split строго по пациентам, метрики AUROC/AUPRC плюс sensitivity/specificity.  
Аугментации держи умеренными: flip, небольшой rotate, brightness/contrast, без агрессивных elastic/deformation; зафиксируй seed, параметры и чекпоинт, чтобы результат был воспроизводимым и этичным.


**Студент:** Окей, попробую начать с ChestX-ray14 или RSNA и ViT-B на 224 px, это пока по силам.  
Только я сомневаюсь: для мультиклассовой разметки ChestX-ray14 лучше считать AUROC/AUPRC по каждому классу, а не общую точность?  
И на первом прогоне допустимо пропустить сложное кропирование и оставить flip + небольшой rotate?


**Научный руководитель:** Да, для ChestX-ray14 считай per-class AUROC/AUPRC и macro-averaged значения, потому что общая accuracy часто обманчива из-за сильного дисбаланса классов.  
На первом прогоне допустимо пропустить сложное кропирование и оставить flip + небольшой rotate, но лучше добавить brightness/contrast и строгий split по пациентам, а не по снимкам.  
Для выбора порога ориентируйся на AUPRC/F1 по классам, а не только на точность, и фиксируй эти метрики в отчёте.


**Студент:** Понял, тогда для первой версии сделаю patient-level split, flip/rotate/brightness и per-class AUROC/AUPRC + macro.  
Осталось непонятно: если классов много и некоторые редкие, стоит ли сразу использовать weighted BCE, а не plain BCE?  
И можно ли сначала прогнать 1 эпоху как smoke-test, прежде чем запускать длинный training?


**Научный руководитель:** Сначала запусти plain BCE как baseline, а weighted BCE/pos_weight сделай отдельным аблационным прогоном: так ты сможешь увидеть, помогает ли взвешивание редким классам, а не сразу усложняешь сравнение.  
Да, 1 эпоху или даже часть батчей можно прогнать как smoke-test, но на ChestX-ray14 лучше проверить на небольшом сабсете, чтобы убедиться, что loss считается, метрики по классам работают, split по пациентам не утекает и память/время предсказуемы.  
После этого запускай полноценный training, фиксируя plain и weighted варианты как отдельные эксперименты.


**Студент:** Хорошо, сделаю plain BCE как baseline и smoke-test на сабсете, чтобы loss и per-class AUPRC не сломались.  
Только я пока не уверен, какой размер сабсета брать: 10–20 пациентов на класс или больше?  
И после smoke-test сразу фиксировать seed, параметры и чекпоинт?


**Научный руководитель:** Для smoke-test не нужно 10–20 пациентов на класс: достаточно мини-сплита в 50–100 train и 20–50 val/test изображений, чтобы покрыть все классы и проверить patient-level split, а не оценить качество.  
Если редких классов мало, включи хотя бы 3–5 их изображений в train/val, а для настоящего эксперимента позже сделай полноценный стратифицированный split.  
Да, сразу после smoke-test фиксируй seed, параметры, split, чекпоинт, loss/metrics и версию кода — это минимальная научная гигиена.


---
*Диалог завершён*

---

## Упражнение для самостоятельной работы

Попробуйте создать свою пару агентов! Несколько идей:

- **Физик-теоретик vs Физик-экспериментатор** — обсуждают, что важнее: теория или эксперимент
- **Преподаватель vs Родитель** — обсуждают ИИ в образовании
- **Историк vs Футуролог** — дебаты о будущем цивилизации

Подберите интересные системные промпты и запустите диалог! Поэкспериментируйте с разными моделями!

In [ ]:
# Ваш код здесь:
# Пример (можно раскомментировать и запустить):
#
# physicist_th = Agent(
#     name="Теоретик",
#     instruction="Ты физик-теоретик. Считаешь, что теория — основа науки. Эксперимент лишь подтверждает. Говори кратко.",
#     model=model_qwen,
# )
# physicist_exp = Agent(
#     name="Экспериментатор",
#     instruction="Ты физик-экспериментатор. Без данных теория — фантазия. Настаивай на измерении. Говори кратко.",
#     model=model_qwen,
# )
# run_dialogue(physicist_th, physicist_exp, topic="Что важнее в физике: теория или эксперимент?", num_turns=3)

print("Здесь можно создать свою пару агентов и запустить диалог.")

---

## Выводы

В этой работе мы:

1. **Создали класс Agent** — обёртку над Responses API с поддержкой контекста через `previous_response_id`

2. **Увидели силу системных промптов** — одна и та же модель с разными инструкциями ведёт себя совершенно по-разному

3. **Организовали диалог двух агентов** — каждый агент получает реплику другого как входное сообщение

### Ключевые моменты

- **`instructions`** определяет роль и поведение агента
- **`previous_response_id`** сохраняет контекст разговора на стороне сервера
- **`store=True`** необходим для сохранения контекста